# DeepLOB "Super Model" (DeepLOB Encoder + Seq2Seq Decoder)

This notebook implements the architecture inspired by the MHF paper.

**Architecture:**
1.  **Encoder:** DeepLOB (Convolutional Blocks + Inception Module + LSTM) to extract rich spatial features from the LOB.
2.  **Decoder:** Seq2Seq LSTM with Additive Attention to generate the 60-second midprice path step-by-step.


In [1]:
import numpy as np
from pathlib import Path
import numpy as np
import torch
import random
from utils.utils import *
from data.simulate_walk_the_book import *
from utils.datastuff import TrainCfg
from utils.train import train_val


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

# Fix randomness for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


device: cuda


In [2]:
# Paths and volume_to_fill
root = Path("data") if Path("data").exists() else Path.cwd()
import sys
if str(root / "src") not in sys.path:
    sys.path.append(str(root / "src"))

symbol_dir = root / "BTCUSDT"
X_path = symbol_dir / "X_train.parquet"
Y_path = symbol_dir / "y_train.parquet"
vol_path = symbol_dir / "vol_to_fill.txt"

volume_to_fill = None
if vol_path.exists():
    import re
    with open(vol_path) as f:
        m = re.search(r"([\d.]+)", f.read())
    if m:
        volume_to_fill = float(m.group(1))
print("volume_to_fill:", volume_to_fill)


volume_to_fill: 4.0


In [3]:
# DeepLOB only take LOB features as input
LOB_COLS = []
for i in range(1, 6):
    LOB_COLS.append(f"ask_price_{i}")
    LOB_COLS.append(f"ask_vol_{i}")
    LOB_COLS.append(f"bid_price_{i}")
    LOB_COLS.append(f"bid_vol_{i}")

FEATURE_COLS = LOB_COLS

# Verification print
print(f"CNN Input Width: 4 (Columns: Price/Vol/Price/Vol)")
print(f"CNN Input Height: 5 (Rows: Levels 1 through 5)")
print(f"Total Features: {len(FEATURE_COLS)}")

CNN Input Width: 4 (Columns: Price/Vol/Price/Vol)
CNN Input Height: 5 (Rows: Levels 1 through 5)
Total Features: 20


In [4]:
# --- Execution ---
config = TrainCfg(

    # Hyperparameters
    epochs = 50,
    batch_size = 32,
    lr = 1e-3,
    weight_decay = 1e-5,
    smooth_lambda = 0.01,
    
    # Windows
    input_window = 600,   # Look-back
    target_window = 60,   # Horizon
    val_ratio = 0.2,

    # Environment
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu"),
    
    # Features & Data
    x_path = X_path,
    y_path = Y_path,
    feature_cols = FEATURE_COLS,
)

model, scalers, val_loader, va_id_map, processor = train_val(config)

Epoch 01 | Train: 0.242064 | Val: 0.472553
Epoch 02 | Train: 0.050348 | Val: 0.117944
Epoch 03 | Train: 0.033587 | Val: 0.015312
Epoch 04 | Train: 0.032329 | Val: 0.025521
Epoch 05 | Train: 0.041625 | Val: 0.030601
Epoch 06 | Train: 0.023610 | Val: 0.066969
Epoch 07 | Train: 0.033225 | Val: 0.011163
Epoch 08 | Train: 0.021769 | Val: 0.007621
Epoch 09 | Train: 0.019124 | Val: 0.009949
Epoch 10 | Train: 0.015899 | Val: 0.030807
Epoch 11 | Train: 0.015867 | Val: 0.011347
Epoch 12 | Train: 0.033261 | Val: 0.882425
Epoch 13 | Train: 0.053272 | Val: 0.089166
Epoch 14 | Train: 0.034024 | Val: 0.018008
Epoch 15 | Train: 0.025427 | Val: 0.025858
Epoch 16 | Train: 0.022479 | Val: 0.021946
Epoch 17 | Train: 0.022207 | Val: 0.011880
Epoch 18 | Train: 0.013559 | Val: 0.015734
Epoch 19 | Train: 0.010139 | Val: 0.003855
Epoch 20 | Train: 0.010068 | Val: 0.020329
Epoch 21 | Train: 0.010494 | Val: 0.032436
Epoch 22 | Train: 0.013801 | Val: 0.026798
Epoch 23 | Train: 0.009641 | Val: 0.045950
Epoch 24 | 

In [5]:
# Implementation error for the model

import pandas as pd
import numpy as np
from data.simulate_walk_the_book import simulate_walk_the_book

# --- 1. Generate predictions using val_loader ---
model.eval()
all_preds = []

with torch.no_grad():
    for xb, yb, target in val_loader:
        xb = xb.to(config.device)
        pred = model(xb, y_teacher=None)
        all_preds.append(pred.cpu().numpy())

val_preds = np.concatenate(all_preds, axis=0)  # Shape: [num_val_ids, 60]

# Get validation IDs
val_ids = [int(uid) for idx, uid in va_id_map.cpu().numpy()]
print(f"Validation predictions shape: {val_preds.shape}")
print(f"Number of validation instruments: {len(val_ids)}")

# --- 2. Reload raw Y validation data to simulate walking the book ---
Y_raw = pd.read_parquet(Y_path).sort_values(["anonymized_id", "time_in_hour"])
Y_val_raw = Y_raw[Y_raw["anonymized_id"].isin(val_ids)].copy()

# --- 3. Column definitions ---
ASK_PRICE_COLS = ['ask_price_1', 'ask_price_2', 'ask_price_3', 'ask_price_4', 'ask_price_5']
ASK_VOL_COLS = ['ask_vol_1', 'ask_vol_2', 'ask_vol_3', 'ask_vol_4', 'ask_vol_5']
BID_PRICE_COLS = ['bid_price_1', 'bid_price_2', 'bid_price_3', 'bid_price_4', 'bid_price_5']
BID_VOL_COLS = ['bid_vol_1', 'bid_vol_2', 'bid_vol_3', 'bid_vol_4', 'bid_vol_5']

# --- 4. Calculate MODEL implementation error ---
model_bps = []

for i, anon_id in enumerate(val_ids):
    df_inst = Y_val_raw[Y_val_raw["anonymized_id"] == anon_id].sort_values("time_in_hour")
    
    if len(df_inst) != 60:
        continue
    
    ask_prices = df_inst[ASK_PRICE_COLS].to_numpy()
    ask_vols = df_inst[ASK_VOL_COLS].to_numpy()
    bid_prices = df_inst[BID_PRICE_COLS].to_numpy()
    bid_vols = df_inst[BID_VOL_COLS].to_numpy()
    close_price = df_inst['close'].dropna().iloc[-1]
    
    # Model-based positions
    mid_preds = val_preds[i]
    pred_close = mid_preds[-1]
    
    price_advantage = pred_close - mid_preds
    price_advantage = np.maximum(price_advantage, 0)
    
    if price_advantage.sum() > 0:
        weights = price_advantage / price_advantage.sum()
    else:
        weights = np.zeros(60)
        weights[-14:] = 1.0 / 14  # Fallback to TWAP
    
    model_positions = weights * volume_to_fill
    
    # Simulate
    model_vol, model_avg_price = simulate_walk_the_book(
        model_positions, ask_prices, ask_vols, bid_prices, bid_vols
    )
    
    if model_vol > 0 and not np.isnan(model_avg_price):
        impl_error = np.abs(model_avg_price - close_price) / close_price * 10000
        vol_penalty = min(100.0, volume_to_fill / model_vol)
        model_bps.append(impl_error * vol_penalty)

model_bps = np.array(model_bps)

print(f"\n{'='*50}")
print(f"MODEL IMPLEMENTATION ERROR")
print(f"{'='*50}")
print(f"Instruments evaluated: {len(model_bps)}")
print(f"Mean:   {model_bps.mean():.4f} bps")
print(f"Median: {np.median(model_bps):.4f} bps")
print(f"Std:    {model_bps.std():.4f} bps")
print(f"Min:    {model_bps.min():.4f} bps")
print(f"Max:    {model_bps.max():.4f} bps")
print(f"{'='*50}")

Validation predictions shape: (136, 60)
Number of validation instruments: 136

MODEL IMPLEMENTATION ERROR
Instruments evaluated: 136
Mean:   1.7226 bps
Median: 1.3524 bps
Std:    1.4364 bps
Min:    0.0000 bps
Max:    8.7199 bps


/root/data/simulate_walk_the_book.py:52: UserWarning: NaN price or volume encountered in ASK book at second 55 level 5. Skipping level.
  warnings.warn(
/root/data/simulate_walk_the_book.py:52: UserWarning: NaN price or volume encountered in ASK book at second 21 level 3. Skipping level.
  warnings.warn(
/root/data/simulate_walk_the_book.py:52: UserWarning: NaN price or volume encountered in ASK book at second 21 level 4. Skipping level.
  warnings.warn(
/root/data/simulate_walk_the_book.py:52: UserWarning: NaN price or volume encountered in ASK book at second 21 level 5. Skipping level.
  warnings.warn(
/root/data/simulate_walk_the_book.py:52: UserWarning: NaN price or volume encountered in ASK book at second 48 level 4. Skipping level.
  warnings.warn(
/root/data/simulate_walk_the_book.py:52: UserWarning: NaN price or volume encountered in ASK book at second 48 level 5. Skipping level.
  warnings.warn(
/root/data/simulate_walk_the_book.py:52: UserWarning: NaN price or volume encounte

In [6]:
# Implementation error for the baseline (TWAP)

K_SECONDS = 14  # TWAP window: last K seconds

baseline_bps = []

for i, anon_id in enumerate(val_ids):
    df_inst = Y_val_raw[Y_val_raw["anonymized_id"] == anon_id].sort_values("time_in_hour")
    
    if len(df_inst) != 60:
        continue
    
    ask_prices = df_inst[ASK_PRICE_COLS].to_numpy()
    ask_vols = df_inst[ASK_VOL_COLS].to_numpy()
    bid_prices = df_inst[BID_PRICE_COLS].to_numpy()
    bid_vols = df_inst[BID_VOL_COLS].to_numpy()
    close_price = df_inst['close'].dropna().iloc[-1]
    
    # Baseline TWAP positions
    baseline_positions = np.zeros(60)
    baseline_positions[-K_SECONDS:] = volume_to_fill / K_SECONDS
    
    # Simulate
    baseline_vol, baseline_avg_price = simulate_walk_the_book(
        baseline_positions, ask_prices, ask_vols, bid_prices, bid_vols
    )
    
    if baseline_vol > 0 and not np.isnan(baseline_avg_price):
        impl_error = np.abs(baseline_avg_price - close_price) / close_price * 10000
        vol_penalty = min(100.0, volume_to_fill / baseline_vol)
        baseline_bps.append(impl_error * vol_penalty)

baseline_bps = np.array(baseline_bps)

print(f"\n{'='*50}")
print(f"BASELINE (TWAP-{K_SECONDS}s) IMPLEMENTATION ERROR")
print(f"{'='*50}")
print(f"Instruments evaluated: {len(baseline_bps)}")
print(f"Mean:   {baseline_bps.mean():.4f} bps")
print(f"Median: {np.median(baseline_bps):.4f} bps")
print(f"Std:    {baseline_bps.std():.4f} bps")
print(f"Min:    {baseline_bps.min():.4f} bps")
print(f"Max:    {baseline_bps.max():.4f} bps")
print(f"{'='*50}")

/root/data/simulate_walk_the_book.py:52: UserWarning: NaN price or volume encountered in ASK book at second 55 level 5. Skipping level.
  warnings.warn(
/root/data/simulate_walk_the_book.py:52: UserWarning: NaN price or volume encountered in ASK book at second 48 level 3. Skipping level.
  warnings.warn(
/root/data/simulate_walk_the_book.py:52: UserWarning: NaN price or volume encountered in ASK book at second 48 level 4. Skipping level.
  warnings.warn(
/root/data/simulate_walk_the_book.py:52: UserWarning: NaN price or volume encountered in ASK book at second 48 level 5. Skipping level.
  warnings.warn(



BASELINE (TWAP-14s) IMPLEMENTATION ERROR
Instruments evaluated: 136
Mean:   1.3340 bps
Median: 1.1894 bps
Std:    1.1121 bps
Min:    0.0000 bps
Max:    9.0855 bps


/root/data/simulate_walk_the_book.py:52: UserWarning: NaN price or volume encountered in ASK book at second 57 level 3. Skipping level.
  warnings.warn(
/root/data/simulate_walk_the_book.py:52: UserWarning: NaN price or volume encountered in ASK book at second 57 level 4. Skipping level.
  warnings.warn(
/root/data/simulate_walk_the_book.py:52: UserWarning: NaN price or volume encountered in ASK book at second 57 level 5. Skipping level.
  warnings.warn(
/root/data/simulate_walk_the_book.py:52: UserWarning: NaN price or volume encountered in ASK book at second 48 level 4. Skipping level.
  warnings.warn(
/root/data/simulate_walk_the_book.py:52: UserWarning: NaN price or volume encountered in ASK book at second 48 level 5. Skipping level.
  warnings.warn(
/root/data/simulate_walk_the_book.py:52: UserWarning: NaN price or volume encountered in ASK book at second 47 level 3. Skipping level.
  warnings.warn(
/root/data/simulate_walk_the_book.py:52: UserWarning: NaN price or volume encounte

In [7]:
# Implementation error for the predictive scheduler

from predictive_scheduler import build_schedule_from_forecasts, ScheduleConfig

# Configure the scheduler
sched_cfg = ScheduleConfig(
    window=14,       # Only trade in last 14 seconds
    alpha=0.1,       # 10% model, 90% TWAP blend
    price_cap=3.0,   # Cap extreme scores
)

scheduler_bps = []

for i, anon_id in enumerate(val_ids):
    df_inst = Y_val_raw[Y_val_raw["anonymized_id"] == anon_id].sort_values("time_in_hour")
    
    if len(df_inst) != 60:
        continue
    
    ask_prices = df_inst[ASK_PRICE_COLS].to_numpy()
    ask_vols = df_inst[ASK_VOL_COLS].to_numpy()
    bid_prices = df_inst[BID_PRICE_COLS].to_numpy()
    bid_vols = df_inst[BID_VOL_COLS].to_numpy()
    close_price = df_inst['close'].dropna().iloc[-1]
    
    # Use predictive_scheduler to build positions
    mid_preds = val_preds[i]
    scheduler_positions = build_schedule_from_forecasts(
        mid_pred=mid_preds,
        volume_to_fill=volume_to_fill,
        spread_pred=None,  # We don't predict spread
        liq_pred=None,     # We don't predict liquidity
        cfg=sched_cfg
    )
    
    # Simulate
    sched_vol, sched_avg_price = simulate_walk_the_book(
        scheduler_positions, ask_prices, ask_vols, bid_prices, bid_vols
    )
    
    if sched_vol > 0 and not np.isnan(sched_avg_price):
        impl_error = np.abs(sched_avg_price - close_price) / close_price * 10000
        vol_penalty = min(100.0, volume_to_fill / sched_vol)
        scheduler_bps.append(impl_error * vol_penalty)

scheduler_bps = np.array(scheduler_bps)

print(f"\n{'='*50}")
print(f"PREDICTIVE SCHEDULER IMPLEMENTATION ERROR")
print(f"{'='*50}")
print(f"Config: window={sched_cfg.window}, alpha={sched_cfg.alpha}")
print(f"Instruments evaluated: {len(scheduler_bps)}")
print(f"Mean:   {scheduler_bps.mean():.4f} bps")
print(f"Median: {np.median(scheduler_bps):.4f} bps")
print(f"Std:    {scheduler_bps.std():.4f} bps")
print(f"Min:    {scheduler_bps.min():.4f} bps")
print(f"Max:    {scheduler_bps.max():.4f} bps")
print(f"{'='*50}")

/root/data/simulate_walk_the_book.py:52: UserWarning: NaN price or volume encountered in ASK book at second 55 level 5. Skipping level.
  warnings.warn(
/root/data/simulate_walk_the_book.py:52: UserWarning: NaN price or volume encountered in ASK book at second 48 level 3. Skipping level.
  warnings.warn(
/root/data/simulate_walk_the_book.py:52: UserWarning: NaN price or volume encountered in ASK book at second 48 level 4. Skipping level.
  warnings.warn(
/root/data/simulate_walk_the_book.py:52: UserWarning: NaN price or volume encountered in ASK book at second 48 level 5. Skipping level.
  warnings.warn(
/root/data/simulate_walk_the_book.py:52: UserWarning: NaN price or volume encountered in ASK book at second 57 level 3. Skipping level.
  warnings.warn(
/root/data/simulate_walk_the_book.py:52: UserWarning: NaN price or volume encountered in ASK book at second 57 level 4. Skipping level.
  warnings.warn(
/root/data/simulate_walk_the_book.py:52: UserWarning: NaN price or volume encounte


PREDICTIVE SCHEDULER IMPLEMENTATION ERROR
Config: window=14, alpha=0.1
Instruments evaluated: 136
Mean:   1.3130 bps
Median: 1.2050 bps
Std:    1.1073 bps
Min:    0.0005 bps
Max:    9.6677 bps


/root/data/simulate_walk_the_book.py:52: UserWarning: NaN price or volume encountered in ASK book at second 48 level 4. Skipping level.
  warnings.warn(
/root/data/simulate_walk_the_book.py:52: UserWarning: NaN price or volume encountered in ASK book at second 48 level 5. Skipping level.
  warnings.warn(
/root/data/simulate_walk_the_book.py:52: UserWarning: NaN price or volume encountered in ASK book at second 47 level 3. Skipping level.
  warnings.warn(
/root/data/simulate_walk_the_book.py:52: UserWarning: NaN price or volume encountered in ASK book at second 47 level 4. Skipping level.
  warnings.warn(
/root/data/simulate_walk_the_book.py:52: UserWarning: NaN price or volume encountered in ASK book at second 47 level 5. Skipping level.
  warnings.warn(
/root/data/simulate_walk_the_book.py:52: UserWarning: NaN price or volume encountered in ASK book at second 49 level 3. Skipping level.
  warnings.warn(
/root/data/simulate_walk_the_book.py:52: UserWarning: NaN price or volume encounte

In [8]:
from utils.test import generate_test_predictions

# --- Execution ---
test_preds, test_id_map = generate_test_predictions(model, config, processor, num_ids=5)

Loading test data...
Sampling first 5 IDs for inference...
Preprocessing test data...
Running inference on 5 instruments...


* **The Map:** This `test_id_np` array pairs the model's row index (e.g., Row 5) with the actual asset ID (e.g., BTC).

In [10]:
# Convert test_id_map to numpy (handles both CPU and GPU tensors)
if hasattr(test_id_map, 'cpu'):
    test_id_np = test_id_map.cpu().numpy().astype(np.uint64)
else:
    test_id_np = np.array(test_id_map, dtype=np.uint64)

# Create a quick summary for inspection
print(f"{'Anonymized ID':<25} | {'First Pred (t+1)':<18} | {'Last Pred (t+60)':<18}")
print("-" * 70)

for i in range(len(test_id_np)):  # Look at first 5
    orig_id = test_id_np[i, 1]
    first_p = test_preds[i, 0]
    last_p  = test_preds[i, -1]
    print(f"{str(orig_id):<25} | {first_p:18.6f} | {last_p:18.6f}")

Anonymized ID             | First Pred (t+1)   | Last Pred (t+60)  
----------------------------------------------------------------------
2362602560156466613       |           1.318301 |           1.198909
6737529360277291661       |          -0.133359 |          -0.121096
10833283855534756836      |          -1.692688 |          -1.703469
15546056879937571314      |          -0.456946 |          -0.477898
16559133294573202526      |           0.325027 |           0.238446


In [11]:
# 1. Load your data into Polars
# Using test_id_np[:, 1] for our big IDs
id_df = pl.DataFrame({
    "anonymized_id": test_id_np[:, 1],
    "preds": test_preds.tolist()  # This creates a column of lists, each 60 items long
})

# 2. Explode and add the time duration
final_df = (
    id_df.explode("preds")  # This turns each list of 60 into 60 separate rows
    .with_columns(
        # This counts 0 to 59 for every ID
        pl.col("anonymized_id").cum_count().over("anonymized_id").sub(1).alias("step")
    )
    .with_columns(
        # Create the duration starting at 59 minutes
        pl.duration(minutes=59, seconds=pl.col("step")).alias("time_in_hour")
    )
    .select([
        "anonymized_id",
        "time_in_hour",
        pl.col("preds").alias("mid_price_pred")
    ])
)

print("✅ Success! Maaz, the table is ready.")
display(final_df.head(65))

✅ Success! Maaz, the table is ready.


anonymized_id,time_in_hour,mid_price_pred
u64,duration[μs],f64
2362602560156466613,59m,1.318301
2362602560156466613,59m 1s,1.317767
2362602560156466613,59m 2s,1.314803
2362602560156466613,59m 3s,1.31257
2362602560156466613,59m 4s,1.310543
…,…,…
6737529360277291661,59m,-0.133359
6737529360277291661,59m 1s,-0.132102
6737529360277291661,59m 2s,-0.131471
